# Tome C1, Chapter 1: Logic and Proof (Python companion)

Exercises first, solutions at the end. We build the tools of the chapter from scratch: formulas as trees, truth tables and normal forms, Tseitin's encoding, resolution, a DPLL satisfiability solver with unit propagation and watched literals, and the linear-time algorithms for Horn clauses and 2-SAT. Then we use the solver: we watch random 3-SAT become hard near the threshold $4.27$, solve Sudoku and the eight queens, and prove two adder circuits equivalent, which is how hardware is verified. Only the standard library, `numpy` and `matplotlib` are needed.

*Before each solution, predict what the output will look like.*

In [ ]:
import itertools, random, time
from collections import defaultdict
import numpy as np
import matplotlib.pyplot as plt
random.seed(1)

## Exercises

**P1.1 (★) Truth tables and normal forms.** Represent a formula as a nested tuple, for example `('and', ('var', 1), ('not', ('var', 2)))`. Write `evaluate`, `truth_table`, and functions building the DNF and the CNF of Theorem 1.3 from a truth table. Check on $200$ random formulas that both normal forms compute the same function. Then check Proposition 1.5 for $n\le 4$: compute the CNF of parity and verify that removing any clause changes the function.

**P1.2 (★★) Tseitin's encoding.** Implement Theorem 1.6: from a formula, produce a CNF (a list of clauses, each a list of non-zero integers as in the DIMACS format, $-v$ meaning $\neg v$) with one new variable per connective. On $100$ random formulas with $5$ variables, check by brute force that every model of the formula extends in exactly one way to a model of the CNF, and that the CNF has at most $4s+1$ clauses.

**P1.3 (★★) A DPLL solver.** Write a DPLL solver with unit propagation driven by *two watched literals per clause* (a clause only needs attention when one of its two watched literals becomes false), chronological backtracking, and branching on the most frequent variable. Test it against brute force on $300$ random 3-CNFs with $12$ variables. Then, for $n=50$ variables and $m=\rho n$ random 3-clauses, plot the fraction of satisfiable formulas and the median number of decisions as $\rho$ goes from $2$ to $7$. Where is the hardest region?

**P1.4 (★★) Resolution.** (a) Implement the Davis–Putnam elimination of the proof of Theorem 1.12 and use it to decide satisfiability of the seven clauses of the opening question. (b) Saturate the nine clauses of $\mathrm{PHP}_2$ under resolution (keeping non-tautological clauses), count the clauses, and extract a refutation tree (Exercise 1.5). (c) Run your DPLL solver on $\mathrm{PHP}_n$ for $n=4,\dots,7$ and count decisions. What growth do you see, and what does Haken's theorem predict?

**P1.5 (★) Horn-SAT in linear time.** Implement the marking algorithm of Theorem 1.16 with a counter per clause (the number of its hypotheses still false) and a queue, so that the running time is linear in the size of the formula. Check it against brute force on random Horn formulas, and check that it returns the least model. Time it on formulas with $10^5$ clauses.

**P1.6 (★★) 2-SAT by strongly connected components.** Build the implication graph, compute its strongly connected components (Kosaraju's two depth-first searches, written iteratively), and decide satisfiability by Theorem 1.17. Extract a model from the components (set $x$ true when the component of $x$ comes after that of $\neg x$ in topological order) and check it. Then estimate the probability that a random 2-CNF with $n=2000$ variables and $m=\rho n$ clauses is satisfiable for $\rho\in[0.5,1.5]$. Where is the threshold?

**P1.7 (★★) Puzzles as SAT.** Encode Sudoku in CNF (variables $x_{r,c,d}$) and solve the puzzle given below, known as one of the hardest for humans. Then encode the $n$ queens problem and count its solutions for $n=6,8$ by repeatedly adding a *blocking clause* that excludes the last solution found. Compare with the known counts $4$ and $92$.

**P1.8 (★★★) CS application: proving two circuits equal.** Build two 8-bit adders as formulas: a ripple-carry adder ($c_{i+1}=a_ib_i\lor c_i(a_i\oplus b_i)$) and a carry-lookahead adder (each carry written directly as $c_{i}=\bigvee_{j<i}\bigl(g_j\land\bigwedge_{j<k<i}p_k\bigr)$ with $g_j=a_jb_j$, $p_k=a_k\oplus b_k$). Form the *miter* "some output bit differs", encode it with Tseitin and show with DPLL that it is unsatisfiable: the circuits agree on all $2^{16}$ inputs. Then plant a bug (replace one $\lor$ by $\land$ in the lookahead adder) and let the solver find an input exposing it.

## Solutions

### P1.1
**How to find it.** A formula is a tree, so every function on formulas is a recursion on the tag of the root. The DNF has one minterm per true row, the CNF one clause per false row.

In [ ]:
def evaluate(f, a):
    # Value of formula f under the assignment a (a dict or list indexed by variable number).
    t = f[0]
    if t == 'var':  return a[f[1]]
    if t == 'const': return f[1]
    if t == 'not':  return 1 - evaluate(f[1], a)
    x, y = evaluate(f[1], a), evaluate(f[2], a)
    return {'and': x & y, 'or': x | y, 'imp': (1 - x) | y, 'iff': int(x == y)}[t]

def variables(f):
    if f[0] == 'var': return {f[1]}
    if f[0] == 'const': return set()
    return set().union(*(variables(g) for g in f[1:]))

def size(f):  # number of connectives
    return 0 if f[0] in ('var', 'const') else 1 + sum(size(g) for g in f[1:])

def truth_table(f, n):
    return tuple(evaluate(f, dict(zip(range(1, n + 1), bits))) for bits in itertools.product((0, 1), repeat=n))

def big(op, fs, empty):
    if not fs: return ('const', empty)
    g = fs[0]
    for h in fs[1:]: g = (op, g, h)
    return g

def lit(i, b):  # the literal x_i if b = 1, not x_i if b = 0
    return ('var', i) if b else ('not', ('var', i))

def dnf(table, n):
    rows = itertools.product((0, 1), repeat=n)
    return big('or', [big('and', [lit(i + 1, b) for i, b in enumerate(r)], 1) for r, v in zip(rows, table) if v], 0)

def cnf_clauses(table, n):  # list of clauses, a clause = list of (variable, sign)
    rows = itertools.product((0, 1), repeat=n)
    return [[(i + 1, 1 - b) for i, b in enumerate(r)] for r, v in zip(rows, table) if not v]

def cnf(table, n):
    return big('and', [big('or', [lit(i, s) for i, s in c], 0) for c in cnf_clauses(table, n)], 1)

def random_formula(n, depth):
    if depth == 0 or random.random() < 0.15:
        return ('var', random.randint(1, n))
    t = random.choice(['not', 'and', 'or', 'imp', 'iff'])
    if t == 'not': return ('not', random_formula(n, depth - 1))
    return (t, random_formula(n, depth - 1), random_formula(n, depth - 1))

n = 4
ok = all(truth_table(dnf(T, n), n) == T == truth_table(cnf(T, n), n)
         for T in (truth_table(random_formula(n, 5), n) for _ in range(200)))
print('DNF and CNF compute the function, 200 random formulas:', ok)

for n in range(1, 5):
    par = tuple(sum(r) % 2 for r in itertools.product((0, 1), repeat=n))
    cl = cnf_clauses(par, n)
    tables = [truth_table(big('and', [big('or', [lit(i, s) for i, s in c], 0) for c in cl[:k] + cl[k + 1:]], 1), n)
              for k in range(len(cl))]
    print(f'n={n}: parity CNF has {len(cl)} = 2^{n-1} clauses; dropping any one changes the function:',
          all(t != par for t in tables))

Dropping a clause of the parity CNF only shows that this particular CNF is irredundant; Proposition 1.5 says more: *no* CNF does better. Its proof is a two-line argument that a computer search could not replace, since there are far too many CNFs to try.

### P1.2
**How to find it.** Number the inner nodes of the parse tree as new variables, then emit for each node the clauses listed in the proof of Theorem 1.6. To check the "unique extension" claim we enumerate assignments of the old variables and, for each, the assignments of the new ones.

In [ ]:
def tseitin(f, nvars):
    # Returns (clauses, root literal, number of variables); new variables start at nvars + 1.
    clauses, counter = [], [nvars]
    def go(g):
        if g[0] == 'var': return g[1]
        kids = [go(h) for h in g[1:]]
        counter[0] += 1; u = counter[0]
        if g[0] == 'not':
            v, = kids; clauses.extend([[u, v], [-u, -v]])
        else:
            v, w = kids
            clauses.extend({'and': [[-u, v], [-u, w], [u, -v, -w]],
                            'or':  [[u, -v], [u, -w], [-u, v, w]],
                            'imp': [[u, v], [u, -w], [-u, -v, w]],
                            'iff': [[-u, -v, w], [-u, v, -w], [u, v, w], [u, -v, -w]]}[g[0]])
        return u
    r = go(f)
    clauses.append([r])
    return clauses, counter[0]

def satisfies(clauses, a):  # a: dict var -> 0/1
    return all(any(a[abs(l)] == (l > 0) for l in c) for c in clauses)

bad = 0
for _ in range(100):
    f = random_formula(5, 4)
    cl, N = tseitin(f, 5)
    assert len(cl) <= 4 * size(f) + 1
    for old in itertools.product((0, 1), repeat=5):
        a = dict(zip(range(1, 6), old))
        ext = [new for new in itertools.product((0, 1), repeat=N - 5)
               if satisfies(cl, {**a, **dict(zip(range(6, N + 1), new))})]
        if len(ext) != evaluate(f, a):   # 1 extension for a model, 0 otherwise
            bad += 1
print('counterexamples found:', bad)

### P1.3
**How to find it.** Unit propagation is where a solver spends its time. With two watched literals per clause, making a literal false only visits the clauses watching it; each such clause either finds another non-false literal to watch, or becomes unit (propagate), or becomes empty (conflict). On backtracking nothing needs to be undone in the watch lists, which is the beauty of the scheme (Moskewicz et al., *Chaff*, 2001).

In [ ]:
class DPLL:
    # A DPLL solver with two watched literals and chronological backtracking.
    def __init__(self, clauses, nvars):
        self.n = nvars
        self.clauses = [list(dict.fromkeys(c)) for c in clauses]
        self.val = [0] * (nvars + 1)              # 0 unassigned, 1 true, -1 false
        self.watches = defaultdict(list)          # literal -> clauses watching it
        self.trail, self.units, self.empty = [], [], False
        self.decisions = self.propagations = 0
        count = defaultdict(int)
        for i, c in enumerate(self.clauses):
            if not c: self.empty = True
            elif len(c) == 1: self.units.append(c[0])
            else:
                self.watches[c[0]].append(i); self.watches[c[1]].append(i)
            for l in c: count[abs(l)] += 1
        self.order = sorted(range(1, nvars + 1), key=lambda v: -count[v])

    def value(self, l):
        v = self.val[abs(l)]
        return v if l > 0 else -v

    def assign(self, l):
        self.val[abs(l)] = 1 if l > 0 else -1
        self.trail.append(l)

    def propagate(self, start):
        # Propagate the literals of the trail from position start; return False on a conflict.
        i = start
        while i < len(self.trail):
            false_lit = -self.trail[i]; i += 1
            ws = self.watches[false_lit]
            j = 0
            while j < len(ws):
                c = self.clauses[ws[j]]
                if c[0] == false_lit: c[0], c[1] = c[1], c[0]   # keep the false watch in position 1
                if self.value(c[0]) == 1: j += 1; continue
                for k in range(2, len(c)):
                    if self.value(c[k]) != -1:                  # found a new literal to watch
                        c[1], c[k] = c[k], c[1]
                        self.watches[c[1]].append(ws[j]); ws[j] = ws[-1]; ws.pop()
                        break
                else:
                    if self.value(c[0]) == -1: return False      # conflict
                    self.assign(c[0]); self.propagations += 1   # unit clause
                    j += 1
        return True

    def solve(self):
        if self.empty: return None
        for l in self.units:
            if self.value(l) == -1: return None
            if self.value(l) == 0: self.assign(l)
        if not self.propagate(0): return None
        stack = []                                # (trail length before decision, literal, flipped?)
        while True:
            v = next((v for v in self.order if self.val[v] == 0), None)
            if v is None:
                return {u: int(self.val[u] == 1) for u in range(1, self.n + 1)}
            self.decisions += 1
            stack.append((len(self.trail), v, False)); start = len(self.trail); self.assign(v)
            while not self.propagate(start):
                while stack and stack[-1][2]:     # undo decisions whose both branches failed
                    pos, _, _ = stack.pop(); self.undo(pos)
                if not stack: return None
                pos, l, _ = stack.pop(); self.undo(pos)
                stack.append((pos, -l, True)); start = len(self.trail); self.assign(-l)

    def undo(self, pos):
        for l in self.trail[pos:]: self.val[abs(l)] = 0
        del self.trail[pos:]

def solve(clauses, nvars):
    s = DPLL(clauses, nvars)
    return s.solve(), s

def random_ksat(n, m, k=3):
    return [[v * random.choice((1, -1)) for v in random.sample(range(1, n + 1), k)] for _ in range(m)]

def brute(clauses, n):
    return any(satisfies(clauses, dict(zip(range(1, n + 1), bits))) for bits in itertools.product((0, 1), repeat=n))

agree = 0
for _ in range(300):
    n = 12; cl = random_ksat(n, random.randint(30, 70))
    model, _ = solve(cl, n)
    if model is not None: assert satisfies(cl, model)
    agree += (model is not None) == brute(cl, n)
print('agreement with brute force on 300 formulas:', agree)

In [ ]:
n, trials = 50, 40
rhos = np.arange(2.0, 7.01, 0.25)
frac, med = [], []
for rho in rhos:
    sat, dec = 0, []
    for _ in range(trials):
        model, s = solve(random_ksat(n, int(rho * n)), n)
        sat += model is not None; dec.append(s.decisions)
    frac.append(sat / trials); med.append(np.median(dec))
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(rhos, frac, 'o-', label='fraction satisfiable')
ax.axvline(4.27, color='gray', ls=':')
ax.set_xlabel('clauses / variables'); ax.set_ylabel('fraction satisfiable')
ax2 = ax.twinx(); ax2.plot(rhos, med, 's-', color='C3', label='median decisions'); ax2.set_ylabel('median decisions', color='C3')
plt.title('Random 3-SAT, n = 50'); plt.show()
print('hardest ratio in this sample:', rhos[int(np.argmax(med))])

Below ratio about $4$ almost all formulas are satisfiable and easy (the solver walks straight to a model); above about $5$ almost all are unsatisfiable, and contradictions appear quickly. The hard ones sit near the threshold, which for large $n$ is believed to be at $\rho\approx4.267$ (Mézard, Parisi and Zecchina, 2002); Ding, Sly and Sun proved the predicted threshold for $k$-SAT with $k$ large (2015). For $n=50$ the transition is still blurred, and the peak of the median work lies near $4$ to $5$.

### P1.4
**How to find it.** Davis–Putnam elimination is the induction step of Theorem 1.12 turned into a loop: pick a variable, replace the clauses containing it by all their resolvents. Saturation keeps adding resolvents until nothing new appears; recording the two parents of each new clause lets us rebuild a refutation tree.

In [ ]:
def davis_putnam(clauses):
    S = {frozenset(c) for c in clauses}
    while True:
        if frozenset() in S: return False
        if not S: return True
        x = min(abs(l) for c in S for l in c)                      # eliminate the smallest variable
        P = [c for c in S if x in c]; N = [c for c in S if -x in c]
        rest = {c for c in S if x not in c and -x not in c}
        for p in P:
            for q in N:
                r = (p - {x}) | (q - {-x})
                if not any(-l in r for l in r): rest.add(r)
        S = rest

opening6 = [[1, 2], [-1, 3], [-2, -3], [-3, 4], [-4, -1], [2, 4]]
print('six clauses satisfiable:', davis_putnam(opening6), '  seven clauses satisfiable:', davis_putnam(opening6 + [[-2, 1]]))

def php(n):  # n+1 pigeons, n holes; variable p(i,j) = n*(i-1) + j
    p = lambda i, j: n * (i - 1) + j
    cl = [[p(i, j) for j in range(1, n + 1)] for i in range(1, n + 2)]
    cl += [[-p(i, j), -p(k, j)] for j in range(1, n + 1) for i in range(1, n + 2) for k in range(i + 1, n + 2)]
    return cl, n * (n + 1)

def saturate(clauses):
    known = {frozenset(c): None for c in clauses}
    while frozenset() not in known:
        new = {}
        for a, b in itertools.combinations(list(known), 2):
            for l in a:
                if -l in b:
                    r = (a - {l}) | (b - {-l})
                    if not any(-x in r for x in r) and r not in known and r not in new:
                        new[r] = (a, b, abs(l))
        if not new: break
        known.update(new)
    return known

cl2, _ = php(2)
sat2 = saturate(cl2)
print('PHP_2: 9 clauses, saturation reached', len(sat2), 'clauses when the empty clause appeared')
def tree_size(c):
    return 0 if sat2[c] is None else 1 + tree_size(sat2[c][0]) + tree_size(sat2[c][1])
print('resolutions in the refutation tree:', tree_size(frozenset()))

for n in range(4, 8):
    cl, N = php(n)
    t = time.time(); model, s = solve(cl, N)
    print(f'PHP_{n}: {len(cl):4d} clauses, satisfiable={model is not None}, decisions={s.decisions:7d}, time {time.time()-t:.2f}s')

The number of decisions grows roughly by a factor $n$ at each step, like $n!$: DPLL tries all ways of placing the pigeons that unit propagation cannot rule out. By Exercise 1.12, a DPLL run is a tree-like resolution refutation, so Haken's theorem forbids any DPLL strategy, however clever its branching, from being polynomial on $\mathrm{PHP}_n$. A human sees the contradiction by counting, a step resolution cannot take; proof systems that can count (cutting planes) refute $\mathrm{PHP}_n$ in polynomial size.

### P1.5
**How to find it.** The naive loop of Theorem 1.16 rescans all clauses after each change. Instead, keep for each definite clause the number of hypotheses not yet true, and for each variable the list of clauses where it is a hypothesis. When a variable becomes true, decrement those counters; a counter reaching $0$ fires the conclusion. Each occurrence of a variable is then handled once.

In [ ]:
def horn_sat(clauses, n):
    # clauses: lists of literals with at most one positive. Returns the least model or None.
    count, concl, occurs, queue = [], [], defaultdict(list), []
    true = [False] * (n + 1)
    for i, c in enumerate(clauses):
        hyps = [-l for l in c if l < 0]; pos = [l for l in c if l > 0]
        count.append(len(hyps)); concl.append(pos[0] if pos else None)
        for h in hyps: occurs[h].append(i)
        if not hyps:
            if pos is None or not pos: return None              # the empty clause
            queue.append(pos[0])
    while queue:
        v = queue.pop()
        if true[v]: continue
        true[v] = True
        for i in occurs[v]:
            count[i] -= 1
            if count[i] == 0:
                if concl[i] is None: return None                 # a goal clause with all hypotheses true
                queue.append(concl[i])
    return {v: int(true[v]) for v in range(1, n + 1)}

def random_horn(n, m):
    cl = []
    for _ in range(m):
        k = random.randint(1, 3)
        hyps = random.sample(range(1, n + 1), k)
        c = [-h for h in hyps[:-1]] + ([hyps[-1]] if random.random() < 0.8 else [-hyps[-1]])
        cl.append(c)
    return cl

ok = least = nsat = 0
for _ in range(300):
    n = 10; cl = random_horn(n, random.randint(5, 25))
    mu = horn_sat(cl, n)
    models = [dict(zip(range(1, n + 1), b)) for b in itertools.product((0, 1), repeat=n)
              if satisfies(cl, dict(zip(range(1, n + 1), b)))]
    ok += (mu is None) == (not models)
    if mu is not None:
        assert satisfies(cl, mu); nsat += 1
        least += all(all(m[v] >= mu[v] for v in mu) for m in models)
print('agreement with brute force:', ok, '/ 300;  least model in', least, 'of the', nsat, 'satisfiable cases')

for m in (10**4, 10**5, 4 * 10**5):
    cl = random_horn(m // 2, m)
    t = time.time(); horn_sat(cl, m // 2); print(f'{m:7d} clauses: {time.time() - t:.3f}s')

The time grows linearly with the number of clauses, as Dowling and Gallier proved. The least model is what a Datalog engine computes: the facts derivable from the rules, and nothing more.

### P1.6
**How to find it.** Kosaraju's algorithm: a first depth-first search records vertices by finishing time; a second search on the reversed graph, in decreasing finishing time, peels off one strongly connected component at a time, in topological order of the condensation. A recursive DFS would overflow Python's stack on $4000$ vertices, so we write it with an explicit stack.

In [ ]:
def scc(nv, adj):
    # Kosaraju; vertices 0..nv-1. Returns comp[v], numbered in topological order of the condensation.
    radj = [[] for _ in range(nv)]
    for u in range(nv):
        for v in adj[u]: radj[v].append(u)
    seen, order = [False] * nv, []
    for s in range(nv):
        if seen[s]: continue
        seen[s] = True; stack = [(s, 0)]
        while stack:
            u, i = stack.pop()
            if i < len(adj[u]):
                stack.append((u, i + 1)); v = adj[u][i]
                if not seen[v]: seen[v] = True; stack.append((v, 0))
            else:
                order.append(u)
    comp, c = [-1] * nv, 0
    for s in reversed(order):
        if comp[s] != -1: continue
        comp[s] = c; stack = [s]
        while stack:
            u = stack.pop()
            for v in radj[u]:
                if comp[v] == -1: comp[v] = c; stack.append(v)
        c += 1
    return comp

def two_sat(clauses, n):
    node = lambda l: 2 * (abs(l) - 1) + (l < 0)     # x -> 2(x-1), not x -> 2(x-1)+1
    adj = [[] for _ in range(2 * n)]
    for a, b in clauses:
        adj[node(-a)].append(node(b)); adj[node(-b)].append(node(a))
    comp = scc(2 * n, adj)
    if any(comp[2 * i] == comp[2 * i + 1] for i in range(n)): return None
    # Kosaraju numbers components in topological order; x is true when its component comes later.
    return {i + 1: int(comp[2 * i] > comp[2 * i + 1]) for i in range(n)}

print('opening question, six clauses:', two_sat(opening6, 4), '  seven:', two_sat(opening6 + [[-2, 1]], 4))
agree = 0
for _ in range(500):
    n = 8; cl = random_ksat(n, random.randint(4, 16), k=2)
    m = two_sat(cl, n)
    if m: assert satisfies(cl, m)
    agree += (m is not None) == brute(cl, n)
print('agreement with brute force:', agree, '/ 500')

n, trials = 2000, 30
rs = np.arange(0.5, 1.51, 0.1)
ps = [np.mean([two_sat(random_ksat(n, int(r * n), k=2), n) is not None for _ in range(trials)]) for r in rs]
plt.figure(figsize=(6, 3)); plt.plot(rs, ps, 'o-'); plt.axvline(1, color='gray', ls=':')
plt.xlabel('clauses / variables'); plt.ylabel('P(satisfiable)'); plt.title('Random 2-SAT, n = 2000'); plt.show()

Why does "the later component is true" give a model? Kosaraju's second pass finds a component only after every component that can reach it; so if $\mathrm{comp}(x)>\mathrm{comp}(\neg x)$ there is no path $x\leadsto\neg x$, and setting the literals of later components true never forces a false one. This is the proof of Theorem 1.17 with the literal $\ell$ always chosen in the last component. The threshold of random 2-SAT is at $\rho=1$ (Chvátal and Reed; Goerdt, 1992): the curve drops through $\frac12$ near $1$ and gets steeper as $n$ grows.

### P1.7
**How to find it.** Sudoku: $9^3=729$ variables $x_{r,c,d}$; clauses say that each cell has a digit, and that each digit appears at most once in each row, column and box; the givens are unit clauses. Queens: one variable per square, "at least one queen per row" clauses, and "at most one" clauses for rows, columns and diagonals. To count solutions, add after each solution the clause "not exactly this set of queens".

In [ ]:
puzzle = ("8........"
          "..36....."
          ".7..9.2.."
          ".5...7..."
          "....457.."
          "...1...3."
          "..1....68"
          "..85...1."
          ".9....4..")
X = lambda r, c, d: 81 * r + 9 * c + d + 1
cl = []
for r in range(9):
    for c in range(9):
        cl.append([X(r, c, d) for d in range(9)])
        cl += [[-X(r, c, d), -X(r, c, e)] for d in range(9) for e in range(d + 1, 9)]
units = [[(r, c) for c in range(9)] for r in range(9)] + [[(r, c) for r in range(9)] for c in range(9)] + \
        [[(3 * i + a, 3 * j + b) for a in range(3) for b in range(3)] for i in range(3) for j in range(3)]
for u in units:
    for d in range(9):
        cl += [[-X(*p, d), -X(*q, d)] for p, q in itertools.combinations(u, 2)]
        cl.append([X(*p, d) for p in u])
for k, ch in enumerate(puzzle):
    if ch != '.': cl.append([X(k // 9, k % 9, int(ch) - 1)])
t = time.time(); model, s = solve(cl, 729)
grid = [[next(d + 1 for d in range(9) if model[X(r, c, d)]) for c in range(9)] for r in range(9)]
print(f'{len(cl)} clauses, {s.decisions} decisions, {time.time()-t:.2f}s')
for row in grid: print(' '.join(map(str, row)))
assert all(sorted(grid[r][c] for r, c in u) == list(range(1, 10)) for u in units)

def queens_count(n):
    Q = lambda r, c: n * r + c + 1
    cl = [[Q(r, c) for c in range(n)] for r in range(n)]
    cells = [(r, c) for r in range(n) for c in range(n)]
    for (r, c), (s_, t_) in itertools.combinations(cells, 2):
        if r == s_ or c == t_ or r - c == s_ - t_ or r + c == s_ + t_:
            cl.append([-Q(r, c), -Q(s_, t_)])
    count = 0
    while True:
        model, _ = solve(cl, n * n)
        if model is None: return count
        count += 1
        cl.append([-v for v in model if model[v]])   # block this solution
for n in (6, 8):
    print(f'{n} queens: {queens_count(n)} solutions')

### P1.8
**How to find it.** Write both adders as formulas over the same input variables, then the miter $\bigvee_i(s_i\oplus s'_i)$, where $s,s'$ are the two sums. The adders are equal exactly when the miter is unsatisfiable. Tseitin turns the miter into a CNF of linear size; equivalence checkers in chip design do exactly this.

In [ ]:
AND = lambda x, y: ('and', x, y); OR = lambda x, y: ('or', x, y)
XOR = lambda x, y: ('not', ('iff', x, y))
def adders(bits=8, bug=False):
    a = [('var', i + 1) for i in range(bits)]; b = [('var', bits + i + 1) for i in range(bits)]
    # ripple carry
    c, s1 = ('const', 0), []
    for i in range(bits):
        s1.append(XOR(XOR(a[i], b[i]), c)); c = OR(AND(a[i], b[i]), AND(c, XOR(a[i], b[i])))
    s1.append(c)
    # carry lookahead: c_i = OR_{j<i} (g_j AND p_{j+1} ... AND p_{i-1})
    g = [AND(a[i], b[i]) for i in range(bits)]; p = [XOR(a[i], b[i]) for i in range(bits)]
    def carry(i):
        terms = []
        for j in range(i):
            t = g[j]
            for k in range(j + 1, i): t = AND(t, p[k])
            terms.append(t)
        f = big('or', terms, 0)
        if bug and i == 5: f = ('and', f[1], f[2])   # the planted bug: one OR becomes AND
        return f
    s2 = [XOR(p[i], carry(i)) for i in range(bits)] + [carry(bits)]
    return s1, s2

def strip_consts(f):
    # Remove constants, as assumed in Theorem 1.6 (enough for these circuits).
    if f[0] in ('var', 'const'): return f
    kids = [strip_consts(g) for g in f[1:]]
    if f[0] == 'not':
        k, = kids
        return ('const', 1 - k[1]) if k[0] == 'const' else ('not', k)
    x, y = kids
    if x[0] == 'const' or y[0] == 'const':
        if x[0] == 'const' and y[0] == 'const': return ('const', evaluate((f[0], x, y), {}))
        cst, other = (x, y) if x[0] == 'const' else (y, x)
        if f[0] == 'and': return other if cst[1] else ('const', 0)
        if f[0] == 'or':  return ('const', 1) if cst[1] else other
        if f[0] == 'iff': return other if cst[1] else ('not', other)
        raise ValueError
    return (f[0], x, y)

for bug in (False, True):
    s1, s2 = adders(8, bug)
    miter = strip_consts(big('or', [XOR(x, y) for x, y in zip(s1, s2)], 0))
    cl, N = tseitin(miter, 16)
    t = time.time(); model, s = solve(cl, N)
    print(f'bug={bug}: miter CNF has {len(cl)} clauses on {N} variables; '
          f'{"equivalent" if model is None else "differ"} ({s.decisions} decisions, {time.time()-t:.2f}s)')
    if model:
        A = sum(model[i + 1] << i for i in range(8)); B = sum(model[9 + i] << i for i in range(8))
        val = lambda S: sum(evaluate(f, model) << i for i, f in enumerate(S))
        print(f'   counterexample: {A} + {B} = {A+B}, ripple gives {val(s1)}, buggy lookahead gives {val(s2)}')

Testing all $2^{16}$ input pairs would also work here, but not for $64$-bit adders ($2^{128}$ pairs), while the miter stays a formula of polynomial size and modern solvers prove such equivalences routinely. Our small DPLL solver struggles as the width grows, because it does not learn clauses; the notebooks of later chapters keep using it, and Tome C3 explains why no solver can be fast on every formula unless P = NP.